# Missing Data Is Not Empty Space
## Statistical analysis of an irregular blood-pressure tracker

This executable notebook treats the **observation process** as part of the statistical problem. The public repository contains only a privacy-safe, day-indexed aggregate snapshot; private source rows are never committed. This is a statistical case study, not medical advice.

The analysis now also asks whether conclusions change after adjusting for **ambient temperature in Fiães at the actual blood-pressure measurement times**.

## 1. Load public data and tested modules

The notebook is a narrative layer over regression-tested Python modules.

In [ ]:
from pathlib import Path
import json
import analysis as primary
import observation_process_sensitivity as observation_sensitivity
import gap_aware_trend_decomposition as gap_aware
import day_influence_sensitivity as influence
import episode_observation_sensitivity as episode_observation
import episode_time_form_sensitivity as time_form
import temporal_dependence_diagnostics as temporal

root = Path.cwd()
data_path = root / 'data/analysis_snapshot.csv'
if not data_path.exists():
    raise FileNotFoundError('Run from Blood_Pressure_Missingness/')
records = primary.load_snapshot(data_path)
observed = primary.observed_records(records)
print(f'{len(records)} calendar days; {len(observed)} observed; {len(records)-len(observed)} missing')

## 2. Observation design first

The current public snapshot has **26 observed days across 65 calendar days**, with a **32-day uninterrupted gap**. Sampling intensity is highly uneven. Raw readings therefore cannot be treated as exchangeable observations from a uniformly observed time series.

## 3. Global trend

In [ ]:
global_fit = primary.linear_trend(records, 'mean_systolic_mmHg')
print(f"Global systolic trend: {global_fit['slope_per_30_days']:.2f} mmHg/30d; 95% CI [{global_fit['ci95_low_per_30_days']:.2f}, {global_fit['ci95_high_per_30_days']:.2f}]")

## 4. Sampling-intensity sensitivity

In [ ]:
observed_days = observation_sensitivity.load_observed_days(data_path)
obs_results = observation_sensitivity.fit_sensitivity_models(observed_days)
for item in obs_results['trend_estimates']:
    print(f"{item['name']}: {item['slope_per_30_days']:.2f} mmHg/30d; 95% CI [{item['ci95_low_per_30_days']:.2f}, {item['ci95_high_per_30_days']:.2f}]")

## 5. Gap-aware decomposition

The global slope should not be read as a smooth trajectory through a 32-day interval with no measurements.

In [ ]:
gap_results = gap_aware.analyze_gap_aware_trend(records)
centered = gap_results['episode_centered_model']
decomp = gap_results['exact_global_slope_decomposition']
print(f"Post-minus-pre contrast: {centered['post_minus_pre_mean_difference_mmHg']:.2f} mmHg")
print(f"Between-episode share of covariance: {100*decomp['between_fraction_of_time_pressure_covariance']:.1f}%")

## 6. Single-day influence

In [ ]:
influence_results = influence.summarize_influence(records)
summary = influence_results['leave_one_day_out_summary']
print(f"Global slope deletion range: [{summary['global_slope_min_per_30_days']:.2f}, {summary['global_slope_max_per_30_days']:.2f}] mmHg/30d")

## 7. Episode-level sampling sensitivity

In [ ]:
episode_obs = episode_observation.fit_episode_observation_sensitivity(records)
for item in episode_obs['estimates']:
    print(f"{item['name']}: {item['episode_difference_mmHg']:.2f} mmHg")

## 8. Within-episode functional-form sensitivity

In [ ]:
time_results = time_form.fit_episode_time_form_sensitivity(records)
for item in time_results['estimates']:
    print(f"{item['name']}: {item['episode_difference_mmHg']:.2f} mmHg")

## 9. Temporal dependence with real calendar spacing

In [ ]:
temporal_results = temporal.diagnose_temporal_dependence(records)
spacing = temporal_results['observed_order_spacing']
print('Observed-row spacing counts:', spacing['calendar_gap_days_counts'])

## 10. Time-matched ambient temperature in Fiães

For each private reading, the refresh workflow retrieves hourly **2 m air temperature for Fiães, Santa Maria da Feira, Portugal** and matches it to the actual local measurement time.

For observed day $d$,

$$T_d=\frac{1}{n_d}\sum_{i=1}^{n_d}T(t_{di}),$$

so $T_d$ is the mean temperature at the **times readings were taken**, not the daily meteorological mean.

The primary exploratory adjustment is

$$Y_d=\beta_0+\beta_1d+\beta_2(T_d-\bar T)+\varepsilon_d,$$

with equal weight per observed day and HC3 robust covariance. The temperature coefficient is an association, not a causal effect.

The private timestamp-weather join remains in memory. Calendar dates, measurement times, matched temperatures, and a day-indexed temperature sequence are not published.

### 10.1 Primary temperature-adjusted model

The secret-backed refresh writes only aggregate diagnostics to `figures/temperature_covariate.json`.

In [ ]:
temperature_path = root / 'figures/temperature_covariate.json'
if temperature_path.exists():
    temperature_result = json.loads(temperature_path.read_text(encoding='utf-8'))
    systolic = temperature_result['model']['metrics']['mean_systolic_mmHg']
    print(f"Adjusted systolic trend: {systolic['day_slope_per_30_days']:.2f} mmHg/30d")
    print(f"Temperature association: {systolic['temperature_coefficient_per_c']:.2f} mmHg/°C")
    print(f"Temperature p-value: {systolic['temperature_p_value']:.4g}")
else:
    print('Run the secret-backed refresh workflow to generate temperature_covariate.json.')

### 10.2 Sensitivity to matching, weighting, and functional form

The robustness layer challenges three choices:

1. nearest-hour matching versus **linear interpolation** between surrounding hourly temperatures;
2. equal-day weighting versus **reading-count weighting**;
3. a linear temperature term versus a **quadratic centered-temperature term**.

We compare both the temperature association and the adjusted 30-day time trend. The goal is specification stability, not selecting the most favorable estimate.

In [ ]:
sensitivity_path = root / 'figures/temperature_covariate_sensitivity.json'
if sensitivity_path.exists():
    sensitivity = json.loads(sensitivity_path.read_text(encoding='utf-8'))
    systolic = sensitivity['metrics']['mean_systolic_mmHg']
    for name, estimate in systolic['specifications'].items():
        print(f"{name}: trend={estimate['day_slope_per_30_days']:.2f} mmHg/30d; temperature={estimate['temperature_coefficient_per_c']:.2f} mmHg/°C")
    print('Direction stability:', systolic['direction_stability'])
else:
    print('Run the secret-backed refresh workflow to generate temperature_covariate_sensitivity.json.')

## 11. Statistical conclusion

The strongest conclusion is not that blood pressure followed a smooth trajectory over the full calendar. The data are dominated by a long unobserved interval and an uneven measurement process.

The current analysis therefore separates several questions: global association, gap-defined episode contrast, day influence, sampling intensity, time-form sensitivity, temporal dependence, and now **ambient temperature at the measurement times**.

Temperature is useful as a plausible contextual covariate, but it does not solve the identification problem. Season, time of day, activity, sleep, medication timing, and the observation process may be related to both temperature and blood pressure. The temperature association is therefore explicitly stress-tested and remains exploratory.

The broader lesson is unchanged but stronger: **missingness and measurement context are part of the statistical process.**